In [ ]:
from google.colab import files

uploaded = files.upload()

Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv
Saving olist_customers_dataset.csv to olist_customers_dataset.csv


In [ ]:
import pandas as pd

orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

datasets = {
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Category translation": category_translation,
    "Customers": customers,
    "Order items": order_items,
    "Payments": payments,
    "Reviews": reviews
}

for name, df in datasets.items():
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("Columns:", list(df.columns))


Orders: 99,441 rows × 8 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Products: 32,951 rows × 9 columns
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Sellers: 3,095 rows × 4 columns
Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Category translation: 71 rows × 2 columns
Columns: ['product_category_name', 'product_category_name_english']

Customers: 99,441 rows × 5 columns
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Order items: 112,650 rows × 7 columns
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

P

In [ ]:
# Check missing values and duplicate rows in each dataset

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Duplicate full rows:", df.duplicated().sum())

    missing = df.isna().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("Missing values: none")
    else:
        print("Missing values by column:")
        print(missing.to_string())


--- Orders ---
Duplicate full rows: 0
Missing values by column:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

--- Products ---
Duplicate full rows: 0
Missing values by column:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

--- Sellers ---
Duplicate full rows: 0
Missing values: none

--- Category translation ---
Duplicate full rows: 0
Missing values: none

--- Customers ---
Duplicate full rows: 0
Missing values: none

--- Order items ---
Duplicate full rows: 0
Missing values: none

--- Payments ---
Duplicate full rows: 0
Missing values: none

--- Reviews ---
Duplicate full rows: 0
Missing values by column:
review_comment_title      88285
review_comment_message    58247


Data-quality checks and handling assumptions

1. Duplicate full rows: No duplicates found in any of the 8 datasets.
2. Orders: Some approval and delivery timestamps are blank. Keep them blank; they may reflect orders that were not approved or delivered, or incomplete records. Exclude missing delivery dates from on-time/late calculations.
3. Products: 610 rows have blank category and related descriptive fields. Keep these values blank rather than guessing a category. Two rows have missing physical measurements; leave them blank.
4. Reviews: Review comment title and message are often blank. Treat these as optional comments, not as missing review scores. Use review_score for rating analysis.
5. No rows are being removed or imputed at this stage. Any analysis that needs a missing field will exclude only the rows missing that field and report the exclusion.

In [ ]:
# Convert the purchase timestamp to a date, then count orders by month

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)

monthly_orders = (
    orders.dropna(subset=["order_purchase_timestamp"])
    .assign(month=lambda df: df["order_purchase_timestamp"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)
    .agg(order_count=("order_id", "nunique"))
    .sort_values("month")
)

print("Monthly order counts:")
display(monthly_orders)

print("\nRows with an unreadable purchase timestamp:",
      orders["order_purchase_timestamp"].isna().sum())

Monthly order counts:


,month,order_count
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026



Rows with an unreadable purchase timestamp: 0


In [ ]:
# Parse delivery dates
date_cols = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Keep orders with both actual and estimated delivery dates
delivery = orders.dropna(subset=date_cols).copy()

# Classify delivery timing
delivery["delivery_performance"] = (
    delivery["order_delivered_customer_date"]
    <= delivery["order_estimated_delivery_date"]
).map({True: "On time", False: "Late"})

# Attach reviews to each order's delivery category
delivery_reviews = reviews.merge(
    delivery[["order_id", "delivery_performance"]],
    on="order_id",
    how="inner"
)

# Summarize ratings by delivery performance
delivery_reviews["low_rating"] = (
    delivery_reviews["review_score"] <= 2
).astype(int)

delivery_summary = (
    delivery_reviews.groupby("delivery_performance", as_index=False)
    .agg(
        review_count=("review_score", "count"),
        average_review_score=("review_score", "mean"),
        low_rating_share=("low_rating", "mean")
    )
)

delivery_summary["low_rating_share"] = (
    delivery_summary["low_rating_share"] * 100
).round(1)

delivery_summary["average_review_score"] = (
    delivery_summary["average_review_score"].round(2)
)

print("Delivery vs. review summary:")
display(delivery_summary)

print(
    "Orders excluded because actual or estimated delivery date is missing:",
    len(orders) - len(delivery)
)

Delivery vs. review summary:


,delivery_performance,review_count,average_review_score,low_rating_share
0,Late,7866,2.55,54.6
1,On time,89147,4.28,9.5


Orders excluded because actual or estimated delivery date is missing: 2965


In [ ]:
print("Delivery-classified orders:")
display(
    delivery.groupby("delivery_performance")
    .agg(order_count=("order_id", "nunique"))
    .reset_index()
)

print("\nReview rows and unique reviewed orders:")
print("Review rows:", len(reviews))
print("Unique order IDs in reviews:", reviews["order_id"].nunique())
print("Review rows with repeated order IDs:", reviews["order_id"].duplicated().sum())

print("\nReview records joined to delivery groups:")
display(
    delivery_reviews.groupby("delivery_performance")
    .agg(
        review_rows=("review_score", "size"),
        unique_orders=("order_id", "nunique")
    )
    .reset_index()
)

print("\nOrders excluded for missing actual or estimated delivery dates:")
print(len(orders) - len(delivery))

Delivery-classified orders:


,delivery_performance,order_count
0,Late,7827
1,On time,88649



Review rows and unique reviewed orders:
Review rows: 100000
Unique order IDs in reviews: 99441
Review rows with repeated order IDs: 559

Review records joined to delivery groups:


,delivery_performance,review_rows,unique_orders
0,Late,7866,7827
1,On time,89147,88649



Orders excluded for missing actual or estimated delivery dates:
2965


In [ ]:
# Check missing delivery dates and confirm the number excluded
print("Orders total:", len(orders))
print("Missing actual delivery date:",
      orders["order_delivered_customer_date"].isna().sum())
print("Missing estimated delivery date:",
      orders["order_estimated_delivery_date"].isna().sum())
print("Orders with both dates:", len(delivery))
print("Orders excluded:", len(orders) - len(delivery))

# Reduce reviews to one average score per order
reviews_per_order = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        average_order_review=("review_score", "mean"),
        review_records=("review_score", "count")
    )
)

# Join one row per order to its delivery classification
order_level = delivery[
    ["order_id", "delivery_performance"]
].merge(
    reviews_per_order,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

# For this summary, a low-rated order has an average score of 2 or less
order_level["low_rated_order"] = (
    order_level["average_order_review"] <= 2
).astype(int)

delivery_summary_order_level = (
    order_level.groupby("delivery_performance", as_index=False)
    .agg(
        orders_with_reviews=("order_id", "nunique"),
        average_review_score=("average_order_review", "mean"),
        low_rated_order_share=("low_rated_order", "mean")
    )
)

delivery_summary_order_level["average_review_score"] = (
    delivery_summary_order_level["average_review_score"].round(2)
)
delivery_summary_order_level["low_rated_order_share_pct"] = (
    delivery_summary_order_level.pop("low_rated_order_share") * 100
).round(1)

display(delivery_summary_order_level)

Orders total: 99441
Missing actual delivery date: 2965
Missing estimated delivery date: 0
Orders with both dates: 96476
Orders excluded: 2965


,delivery_performance,orders_with_reviews,average_review_score,low_rated_order_share_pct
0,Late,7827,2.55,54.6
1,On time,88649,4.28,9.5


In [ ]:
# Map each order to the customer's stable unique ID
orders_with_customer = orders[
    ["order_id", "customer_id"]
].merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Count distinct orders per unique customer
customer_order_counts = (
    orders_with_customer.groupby("customer_unique_id", as_index=False)
    .agg(order_count=("order_id", "nunique"))
)

# Summarize one-time and repeat customers
customer_order_counts["customer_type"] = (
    customer_order_counts["order_count"] >= 2
).map({True: "Repeat", False: "One-time"})

repeat_customer_summary = (
    customer_order_counts.groupby("customer_type", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique")
    )
)

total_customers = customer_order_counts["customer_unique_id"].nunique()
repeat_customers = customer_order_counts.loc[
    customer_order_counts["order_count"] >= 2,
    "customer_unique_id"
].nunique()

print("Unique customers:", f"{total_customers:,}")
print("Repeat customers:", f"{repeat_customers:,}")
print("Repeat customer rate:", f"{repeat_customers / total_customers * 100:.2f}%")
display(repeat_customer_summary)

Unique customers: 96,096
Repeat customers: 2,997
Repeat customer rate: 3.12%


,customer_type,customers
0,One-time,93099
1,Repeat,2997


In [ ]:
import pandas as pd
from google.colab import files
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# 1) Build a data-quality audit table from the 8 loaded datasets
audit_rows = []

for name, df in datasets.items():
    audit_rows.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate full rows": int(df.duplicated().sum()),
        "Missing cells": int(df.isna().sum().sum())
    })

data_quality = pd.DataFrame(audit_rows)

# 2) Prepare a repeat-customer table with counts and percentages
customer_repeat_export = repeat_customer_summary.copy()
customer_repeat_export["share_of_customers_pct"] = (
    customer_repeat_export["customers"] / total_customers * 100
).round(2)

# 3) Create key findings with careful interpretation and caveats
key_findings = pd.DataFrame([
    {
        "Finding": "Order volume increased substantially during 2017",
        "Evidence": "Monthly orders rose from 800 in Jan 2017 to 7,544 in Nov 2017.",
        "Interpretation / caveat": (
            "The series is uneven month to month. Sep–Oct 2016 and Sep–Oct 2018 "
            "are partial months, and Nov 2016 is absent; avoid treating these "
            "edge months as complete-month comparisons."
        )
    },
    {
        "Finding": "Late delivery is associated with lower review scores",
        "Evidence": (
            "Average order review score: 2.55 for late orders vs. 4.28 for on-time orders. "
            "Orders with average score <=2: 54.6% late vs. 9.5% on time."
        ),
        "Interpretation / caveat": (
            "This is an association, not proof that delivery timing alone caused "
            "lower ratings. Analysis includes orders with both actual and estimated "
            "delivery dates and at least one review."
        )
    },
    {
        "Finding": "Repeat-customer share is low in this dataset",
        "Evidence": (
            "2,997 of 96,096 unique customers placed 2 or more orders (3.12%); "
            "93,099 customers placed one order."
        ),
        "Interpretation / caveat": (
            "Customers are identified by customer_unique_id. The dataset ends in "
            "2018, and customers have different lengths of time in which to return; "
            "interpret this as observed repeat purchasing in the available data."
        )
    }
])

# 4) Document assumptions and handling decisions
assumptions = pd.DataFrame([
    ["Duplicates", "No full-row duplicates found in any of the 8 datasets; no duplicate rows removed."],
    ["Missing order timestamps", "Keep missing approval/delivery timestamps blank; do not impute dates."],
    ["Delivery analysis", "Classify only orders with both actual customer delivery date and estimated delivery date. Missing actual delivery dates are excluded."],
    ["Review comments", "Blank review titles/messages are treated as optional comments; use review_score for rating analysis."],
    ["Review join", "Reduce review records to one average score per order before comparing delivery groups, so each order contributes once."],
    ["Low-rated order metric", "An order is counted as low-rated when its average review score is <=2."],
    ["Repeat customers", "Use customer_unique_id to count unique customers and distinct order_id values per customer; repeat means 2 or more orders."],
    ["Monthly trend", "Group orders by order_purchase_timestamp month. Sep–Oct 2016 and Sep–Oct 2018 are partial months; Nov 2016 is absent."]
], columns=["Topic", "Assumption / handling"])

readme = pd.DataFrame([
    ["Purpose", "Olist e-commerce case assessment analysis workbook."],
    ["Contents", "Data quality audit, monthly order trend, delivery and review analysis, repeat-customer summary, key findings, and assumptions."],
    ["Data basis", "Eight supplied Olist CSV datasets."],
    ["Important note", "Findings are descriptive. Associations should not be presented as causal effects."]
], columns=["Item", "Details"])

# 5) Export all sheets into one Excel file
output_file = "PrateekKumar_DTU_Analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    readme.to_excel(writer, sheet_name="Read Me", index=False)
    data_quality.to_excel(writer, sheet_name="Data Quality", index=False)
    monthly_orders.to_excel(writer, sheet_name="Monthly Orders", index=False)
    delivery_summary_order_level.to_excel(writer, sheet_name="Delivery Reviews", index=False)
    customer_repeat_export.to_excel(writer, sheet_name="Customer Repeat", index=False)
    key_findings.to_excel(writer, sheet_name="Key Findings", index=False)
    assumptions.to_excel(writer, sheet_name="Assumptions", index=False)

# 6) Apply readable formatting to each worksheet
from openpyxl import load_workbook

wb = load_workbook(output_file)

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    # Header formatting
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.alignment = Alignment(vertical="center", wrap_text=True)

    # Wrap cell content and set reasonable column widths
    for column_cells in ws.columns:
        col_letter = get_column_letter(column_cells[0].column)
        max_len = 0

        for cell in column_cells:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))

        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 55)

    ws.row_dimensions[1].height = 30

wb.save(output_file)

print("Workbook created:", output_file)
files.download(output_file)

Workbook created: PrateekKumar_DTU_Analysis.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from openpyxl import load_workbook
from openpyxl.chart import LineChart, BarChart, Reference
from google.colab import files
import os

# Workbook created by your earlier export cell
workbook_path = "PrateekKumar_DTU_Analysis.xlsx"

if not os.path.exists(workbook_path):
    raise FileNotFoundError(
        "Workbook not found in this Colab session. "
        "Run your workbook-export cell first, or upload the .xlsx file."
    )

wb = load_workbook(workbook_path)

# Remove existing charts first, so rerunning this cell won't duplicate them
for ws in wb.worksheets:
    ws._charts = []

# --- Chart 1: Monthly order trend ---
ws = wb["Monthly Orders"]

chart = LineChart()
chart.title = "Monthly Orders"
chart.style = 2
chart.y_axis.title = "Number of orders"
chart.x_axis.title = "Month"
chart.height = 8
chart.width = 18
chart.marker = "circle"

data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
categories = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)
chart.legend = None
ws.add_chart(chart, "D2")


# --- Chart 2: Average review score by delivery performance ---
ws = wb["Delivery Reviews"]

chart = BarChart()
chart.type = "col"
chart.style = 10
chart.title = "Average Review Score: Late vs On Time"
chart.y_axis.title = "Average review score (out of 5)"
chart.x_axis.title = "Delivery performance"
chart.height = 8
chart.width = 18

data = Reference(ws, min_col=3, min_row=1, max_row=ws.max_row)
categories = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)
chart.legend = None
ws.add_chart(chart, "F2")


# --- Chart 3: Low-rated order share by delivery performance ---
# Separate chart because its percentage scale differs from the average score.
chart2 = BarChart()
chart2.type = "col"
chart2.style = 11
chart2.title = "Share of Low-Rated Orders (Average Score <= 2)"
chart2.y_axis.title = "Orders (%)"
chart2.x_axis.title = "Delivery performance"
chart2.height = 8
chart2.width = 18

data = Reference(ws, min_col=4, min_row=1, max_row=ws.max_row)
categories = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)

chart2.add_data(data, titles_from_data=True)
chart2.set_categories(categories)
chart2.legend = None
ws.add_chart(chart2, "F20")


# --- Chart 4: One-time vs repeat customers ---
ws = wb["Customer Repeat"]

chart = BarChart()
chart.type = "col"
chart.style = 10
chart.title = "One-Time vs Repeat Customers"
chart.y_axis.title = "Number of customers"
chart.x_axis.title = "Customer type"
chart.height = 8
chart.width = 18

data = Reference(ws, min_col=2, min_row=1, max_row=ws.max_row)
categories = Reference(ws, min_col=1, min_row=2, max_row=ws.max_row)

chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)
chart.legend = None
ws.add_chart(chart, "E2")


# Save updated workbook and download it
wb.save(workbook_path)

print("Charts added. Updated workbook:", workbook_path)
files.download(workbook_path)

Charts added. Updated workbook: PrateekKumar_DTU_Analysis.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, BarChart, PieChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.utils import get_column_letter
from google.colab import files
import os

# Use the workbook created earlier in this Colab session.
# If it is not present, upload your latest edited .xlsx workbook.
workbook_path = "PrateekKumar_DTU_Analysis.xlsx"

if not os.path.exists(workbook_path):
    print("Upload your latest Excel workbook (.xlsx) now.")
    uploaded = files.upload()
    workbook_path = next(
        (name for name in uploaded.keys() if name.lower().endswith(".xlsx")),
        None
    )
    if workbook_path is None:
        raise FileNotFoundError("No .xlsx workbook was uploaded.")

wb = load_workbook(workbook_path)

required_sheets = ["Monthly Orders", "Delivery Reviews", "Customer Repeat"]
missing_sheets = [s for s in required_sheets if s not in wb.sheetnames]
if missing_sheets:
    raise ValueError(f"Workbook is missing required tabs: {missing_sheets}")

# Re-create the dashboard cleanly if this cell is run more than once.
if "Dashboard" in wb.sheetnames:
    del wb["Dashboard"]

dash = wb.create_sheet("Dashboard", 0)
dash.sheet_view.showGridLines = False

# Set dashboard column widths.
for col in range(1, 18):
    dash.column_dimensions[get_column_letter(col)].width = 13

# Colors and reusable formatting.
navy = "17365D"
blue = "1F4E78"
light_blue = "D9EAF7"
pale = "F3F6FA"
dark = "243447"
white = "FFFFFF"
gray = "667085"
border_side = Side(style="thin", color="D5DDE5")

# Main title.
dash.merge_cells("A1:P1")
dash["A1"] = "OLIST E-COMMERCE | PERFORMANCE DASHBOARD"
dash["A1"].font = Font(name="Aptos Display", size=18, bold=True, color=white)
dash["A1"].fill = PatternFill("solid", fgColor=navy)
dash["A1"].alignment = Alignment(horizontal="left", vertical="center")
dash.row_dimensions[1].height = 34

dash.merge_cells("A2:P2")
dash["A2"] = (
    "Descriptive analysis of order volume, delivery performance, customer ratings "
    "and observed repeat purchasing."
)
dash["A2"].font = Font(name="Aptos", size=10, italic=True, color=gray)
dash["A2"].alignment = Alignment(wrap_text=True, vertical="center")
dash.row_dimensions[2].height = 28

# Read summary values from the existing tabs.
monthly_ws = wb["Monthly Orders"]
delivery_ws = wb["Delivery Reviews"]
repeat_ws = wb["Customer Repeat"]

monthly_rows = list(monthly_ws.iter_rows(min_row=2, values_only=True))
monthly_rows = [r for r in monthly_rows if r[0] is not None and r[1] is not None]

delivery_data = {
    str(row[0]).strip(): row
    for row in delivery_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None
}
repeat_data = {
    str(row[0]).strip(): row
    for row in repeat_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None
}

# Validate the expected category labels.
if "Late" not in delivery_data or "On time" not in delivery_data:
    raise ValueError("Delivery Reviews needs 'Late' and 'On time' rows.")
if "One-time" not in repeat_data or "Repeat" not in repeat_data:
    raise ValueError("Customer Repeat needs 'One-time' and 'Repeat' rows.")

late_count = delivery_data["Late"][1]
on_time_count = delivery_data["On time"][1]
late_avg = delivery_data["Late"][2]
on_time_avg = delivery_data["On time"][2]
late_low_pct = delivery_data["Late"][3]
on_time_low_pct = delivery_data["On time"][3]

one_time_customers = repeat_data["One-time"][1]
repeat_customers = repeat_data["Repeat"][1]
repeat_share = repeat_data["Repeat"][2]

total_customers = one_time_customers + repeat_customers
total_delivery_grouped = late_count + on_time_count

# KPI cards: merged label/value cells with clear headings.
kpis = [
    ("A4:D4", "A5:D6", "ORDERS WITH DELIVERY DATES", total_delivery_grouped, "#,##0"),
    ("E4:H4", "E5:H6", "LATE-DELIVERY ORDERS", late_count, "#,##0"),
    ("I4:L4", "I5:L6", "ON-TIME AVG. REVIEW SCORE", on_time_avg, '0.00" / 5"'),
    ("M4:P4", "M5:P6", "OBSERVED REPEAT-CUSTOMER SHARE", repeat_share / 100, "0.00%"),
]

for label_range, value_range, label, value, number_format in kpis:
    dash.merge_cells(label_range)
    dash.merge_cells(value_range)

    label_cell = dash[label_range.split(":")[0]]
    value_cell = dash[value_range.split(":")[0]]

    label_cell.value = label
    label_cell.font = Font(name="Aptos", size=9, bold=True, color=blue)
    label_cell.fill = PatternFill("solid", fgColor=light_blue)
    label_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    value_cell.value = value
    value_cell.number_format = number_format
    value_cell.font = Font(name="Aptos Display", size=20, bold=True, color=navy)
    value_cell.fill = PatternFill("solid", fgColor=pale)
    value_cell.alignment = Alignment(horizontal="center", vertical="center")

    for row in dash[label_range]:
        for cell in row:
            cell.border = Border(left=border_side, right=border_side,
                                 top=border_side, bottom=border_side)
    for row in dash[value_range]:
        for cell in row:
            cell.border = Border(left=border_side, right=border_side,
                                 top=border_side, bottom=border_side)

dash.row_dimensions[4].height = 28
dash.row_dimensions[5].height = 25
dash.row_dimensions[6].height = 25

# Helper to style chart headings in the dashboard.
def add_section_title(cell_ref, text):
    cell = dash[cell_ref]
    cell.value = text
    cell.font = Font(name="Aptos", size=11, bold=True, color=navy)
    cell.alignment = Alignment(vertical="center")
    cell.fill = PatternFill("solid", fgColor=light_blue)

add_section_title("A8", "ORDER VOLUME TREND")
add_section_title("I8", "DELIVERY PERFORMANCE & RATINGS")
add_section_title("A25", "CUSTOMER MIX")
add_section_title("I25", "RATING COMPARISON")

# Chart 1: Monthly order trend — line chart with markers and axis labels.
line = LineChart()
line.title = "Monthly Orders"
line.style = 13
line.y_axis.title = "Number of orders"
line.x_axis.title = "Purchase month"
line.height = 8.2
line.width = 17.5
line.legend = None
line.marker = "circle"

monthly_data = Reference(
    monthly_ws, min_col=2, min_row=1, max_row=monthly_ws.max_row
)
monthly_categories = Reference(
    monthly_ws, min_col=1, min_row=2, max_row=monthly_ws.max_row
)
line.add_data(monthly_data, titles_from_data=True)
line.set_categories(monthly_categories)
line.dataLabels = DataLabelList()
line.dataLabels.showVal = True
line.dataLabels.showLegendKey = False
dash.add_chart(line, "A9")

# Chart 2: Late vs on-time order counts — bar chart with values.
delivery_count_chart = BarChart()
delivery_count_chart.type = "col"
delivery_count_chart.style = 10
delivery_count_chart.title = "Orders by Delivery Status"
delivery_count_chart.y_axis.title = "Orders"
delivery_count_chart.x_axis.title = "Delivery status"
delivery_count_chart.height = 8.2
delivery_count_chart.width = 17.5
delivery_count_chart.legend = None

count_data = Reference(
    delivery_ws, min_col=2, min_row=1, max_row=delivery_ws.max_row
)
delivery_categories = Reference(
    delivery_ws, min_col=1, min_row=2, max_row=delivery_ws.max_row
)
delivery_count_chart.add_data(count_data, titles_from_data=True)
delivery_count_chart.set_categories(delivery_categories)
delivery_count_chart.dataLabels = DataLabelList()
delivery_count_chart.dataLabels.showVal = True
dash.add_chart(delivery_count_chart, "I9")

# Chart 3: Customer mix — pie chart with category and percentage labels.
pie = PieChart()
pie.title = "Observed Customer Mix"
pie.height = 8.2
pie.width = 17.5

customer_data = Reference(
    repeat_ws, min_col=2, min_row=1, max_row=repeat_ws.max_row
)
customer_categories = Reference(
    repeat_ws, min_col=1, min_row=2, max_row=repeat_ws.max_row
)
pie.add_data(customer_data, titles_from_data=True)
pie.set_categories(customer_categories)
pie.dataLabels = DataLabelList()
pie.dataLabels.showPercent = True
pie.dataLabels.showCatName = True
pie.dataLabels.showVal = True
pie.dataLabels.showLeaderLines = True
dash.add_chart(pie, "A26")

# Chart 4: Average review score by delivery status.
avg_chart = BarChart()
avg_chart.type = "col"
avg_chart.style = 11
avg_chart.title = "Average Review Score by Delivery Status"
avg_chart.y_axis.title = "Average score (out of 5)"
avg_chart.x_axis.title = "Delivery status"
avg_chart.height = 8.2
avg_chart.width = 17.5
avg_chart.legend = None

avg_data = Reference(
    delivery_ws, min_col=3, min_row=1, max_row=delivery_ws.max_row
)
avg_chart.add_data(avg_data, titles_from_data=True)
avg_chart.set_categories(delivery_categories)
avg_chart.dataLabels = DataLabelList()
avg_chart.dataLabels.showVal = True
dash.add_chart(avg_chart, "I26")

# Add a brief interpretation/caveat block beneath the charts.
dash.merge_cells("A43:P43")
dash["A43"] = "HOW TO READ THIS DASHBOARD"
dash["A43"].font = Font(name="Aptos", size=11, bold=True, color=white)
dash["A43"].fill = PatternFill("solid", fgColor=blue)
dash["A43"].alignment = Alignment(vertical="center")
dash.row_dimensions[43].height = 24

dash.merge_cells("A44:P46")
dash["A44"] = (
    "Delivery group counts include orders with both actual and estimated delivery dates. "
    "The review comparison is descriptive and does not establish that delivery timing alone "
    "caused a rating difference. Repeat purchasing means 2 or more distinct orders per "
    "customer_unique_id within the available dataset. September–October 2016 and "
    "September–October 2018 are partial months; November 2016 is absent."
)
dash["A44"].font = Font(name="Aptos", size=10, color=dark)
dash["A44"].alignment = Alignment(wrap_text=True, vertical="top")
dash["A44"].fill = PatternFill("solid", fgColor=pale)

# Set dashboard view to open at top-left.
dash.freeze_panes = "A4"
dash.sheet_view.zoomScale = 80
dash.sheet_properties.pageSetUpPr.fitToPage = True
dash.page_setup.orientation = "landscape"
dash.page_setup.fitToWidth = 1
dash.page_setup.fitToHeight = 2
dash.print_area = "A1:P46"

# Make dashboard the first active sheet.
wb.active = 0

# Save as a new file so the previous workbook remains untouched.
dashboard_file = "PrateekKumar_DTU_Analysis_Dashboard.xlsx"
wb.save(dashboard_file)

print("Dashboard added successfully.")
print("Saved workbook:", dashboard_file)
files.download(dashboard_file)

Dashboard added successfully.
Saved workbook: PrateekKumar_DTU_Analysis_Dashboard.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, BarChart, PieChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.utils import get_column_letter
from google.colab import files

# Upload the latest version of your Excel workbook.
print("Upload your latest Excel workbook (.xlsx).")
uploaded = files.upload()

workbook_path = next(
    (name for name in uploaded.keys() if name.lower().endswith(".xlsx")),
    None
)

if workbook_path is None:
    raise FileNotFoundError("No .xlsx workbook was uploaded.")

wb = load_workbook(workbook_path)

required_sheets = ["Monthly Orders", "Delivery Reviews", "Customer Repeat"]
missing_sheets = [s for s in required_sheets if s not in wb.sheetnames]

if missing_sheets:
    raise ValueError(f"Workbook is missing required tabs: {missing_sheets}")

# Re-create the Dashboard cleanly if this cell is run more than once.
if "Dashboard" in wb.sheetnames:
    del wb["Dashboard"]

dash = wb.create_sheet("Dashboard", 0)
dash.sheet_view.showGridLines = False

# Set dashboard column widths.
for col in range(1, 17):
    dash.column_dimensions[get_column_letter(col)].width = 13

# Colors and reusable formatting.
navy = "17365D"
blue = "1F4E78"
light_blue = "D9EAF7"
pale = "F3F6FA"
dark = "243447"
white = "FFFFFF"
gray = "667085"
border_side = Side(style="thin", color="D5DDE5")

# Main title.
dash.merge_cells("A1:P1")
dash["A1"] = "OLIST E-COMMERCE | PERFORMANCE DASHBOARD"
dash["A1"].font = Font(name="Aptos Display", size=18, bold=True, color=white)
dash["A1"].fill = PatternFill("solid", fgColor=navy)
dash["A1"].alignment = Alignment(horizontal="left", vertical="center")
dash.row_dimensions[1].height = 34

dash.merge_cells("A2:P2")
dash["A2"] = (
    "Descriptive analysis of order volume, delivery performance, customer ratings "
    "and observed repeat purchasing."
)
dash["A2"].font = Font(name="Aptos", size=10, italic=True, color=gray)
dash["A2"].alignment = Alignment(wrap_text=True, vertical="center")
dash.row_dimensions[2].height = 28

# Read summary values from the existing tabs.
monthly_ws = wb["Monthly Orders"]
delivery_ws = wb["Delivery Reviews"]
repeat_ws = wb["Customer Repeat"]

monthly_rows = [
    row for row in monthly_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None and row[1] is not None
]

delivery_data = {
    str(row[0]).strip(): row
    for row in delivery_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None
}

repeat_data = {
    str(row[0]).strip(): row
    for row in repeat_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None
}

# Validate the expected category labels.
if "Late" not in delivery_data or "On time" not in delivery_data:
    raise ValueError("Delivery Reviews needs 'Late' and 'On time' rows.")

if "One-time" not in repeat_data or "Repeat" not in repeat_data:
    raise ValueError("Customer Repeat needs 'One-time' and 'Repeat' rows.")

late_count = delivery_data["Late"][1]
on_time_count = delivery_data["On time"][1]
late_avg = delivery_data["Late"][2]
on_time_avg = delivery_data["On time"][2]
late_low_pct = delivery_data["Late"][3]
on_time_low_pct = delivery_data["On time"][3]

one_time_customers = repeat_data["One-time"][1]
repeat_customers = repeat_data["Repeat"][1]
repeat_share = repeat_data["Repeat"][2]

total_customers = one_time_customers + repeat_customers
total_delivery_grouped = late_count + on_time_count

# KPI cards.
kpis = [
    ("A4:D4", "A5:D6", "ORDERS WITH DELIVERY DATES",
     total_delivery_grouped, "#,##0"),
    ("E4:H4", "E5:H6", "LATE-DELIVERY ORDERS",
     late_count, "#,##0"),
    ("I4:L4", "I5:L6", "ON-TIME AVG. REVIEW SCORE",
     on_time_avg, '0.00" / 5"'),
    ("M4:P4", "M5:P6", "OBSERVED REPEAT-CUSTOMER SHARE",
     repeat_share / 100, "0.00%"),
]

for label_range, value_range, label, value, number_format in kpis:
    dash.merge_cells(label_range)
    dash.merge_cells(value_range)

    label_cell = dash[label_range.split(":")[0]]
    value_cell = dash[value_range.split(":")[0]]

    label_cell.value = label
    label_cell.font = Font(name="Aptos", size=9, bold=True, color=blue)
    label_cell.fill = PatternFill("solid", fgColor=light_blue)
    label_cell.alignment = Alignment(
        horizontal="center", vertical="center", wrap_text=True
    )

    value_cell.value = value
    value_cell.number_format = number_format
    value_cell.font = Font(
        name="Aptos Display", size=20, bold=True, color=navy
    )
    value_cell.fill = PatternFill("solid", fgColor=pale)
    value_cell.alignment = Alignment(horizontal="center", vertical="center")

    for row in dash[label_range]:
        for cell in row:
            cell.border = Border(
                left=border_side, right=border_side,
                top=border_side, bottom=border_side
            )

    for row in dash[value_range]:
        for cell in row:
            cell.border = Border(
                left=border_side, right=border_side,
                top=border_side, bottom=border_side
            )

dash.row_dimensions[4].height = 28
dash.row_dimensions[5].height = 25
dash.row_dimensions[6].height = 25

# Helper to style chart headings.
def add_section_title(cell_ref, title_text):
    cell = dash[cell_ref]
    cell.value = title_text
    cell.font = Font(name="Aptos", size=11, bold=True, color=navy)
    cell.alignment = Alignment(vertical="center")
    cell.fill = PatternFill("solid", fgColor=light_blue)

add_section_title("A8", "ORDER VOLUME TREND")
add_section_title("I8", "DELIVERY PERFORMANCE & RATINGS")
add_section_title("A25", "CUSTOMER MIX")
add_section_title("I25", "RATING COMPARISON")

# Chart 1: Monthly order trend.
# Data labels are intentionally OFF to avoid overlapping text.
line = LineChart()
line.title = "Monthly Orders"
line.style = 13
line.y_axis.title = "Number of orders"
line.x_axis.title = "Purchase month"
line.height = 8.2
line.width = 17.5
line.legend = None

monthly_data = Reference(
    monthly_ws,
    min_col=2,
    min_row=1,
    max_row=monthly_ws.max_row
)

monthly_categories = Reference(
    monthly_ws,
    min_col=1,
    min_row=2,
    max_row=monthly_ws.max_row
)

line.add_data(monthly_data, titles_from_data=True)
line.set_categories(monthly_categories)

# Show only every third month on the horizontal axis.
line.x_axis.tickLblSkip = 3
line.x_axis.tickMarkSkip = 3

# Do not show labels on individual monthly data points.
line.dataLabels = None

dash.add_chart(line, "A9")

# Chart 2: Late vs on-time order counts.
delivery_count_chart = BarChart()
delivery_count_chart.type = "col"
delivery_count_chart.style = 10
delivery_count_chart.title = "Orders by Delivery Status"
delivery_count_chart.y_axis.title = "Orders"
delivery_count_chart.x_axis.title = "Delivery status"
delivery_count_chart.height = 8.2
delivery_count_chart.width = 17.5
delivery_count_chart.legend = None

count_data = Reference(
    delivery_ws,
    min_col=2,
    min_row=1,
    max_row=delivery_ws.max_row
)

delivery_categories = Reference(
    delivery_ws,
    min_col=1,
    min_row=2,
    max_row=delivery_ws.max_row
)

delivery_count_chart.add_data(count_data, titles_from_data=True)
delivery_count_chart.set_categories(delivery_categories)
delivery_count_chart.dataLabels = DataLabelList()
delivery_count_chart.dataLabels.showVal = True

dash.add_chart(delivery_count_chart, "I9")

# Chart 3: Customer mix.
pie = PieChart()
pie.title = "Observed Customer Mix"
pie.height = 8.2
pie.width = 17.5

customer_data = Reference(
    repeat_ws,
    min_col=2,
    min_row=1,
    max_row=repeat_ws.max_row
)

customer_categories = Reference(
    repeat_ws,
    min_col=1,
    min_row=2,
    max_row=repeat_ws.max_row
)

pie.add_data(customer_data, titles_from_data=True)
pie.set_categories(customer_categories)
pie.dataLabels = DataLabelList()
pie.dataLabels.showPercent = True
pie.dataLabels.showCatName = True
pie.dataLabels.showVal = True
pie.dataLabels.showLeaderLines = True

dash.add_chart(pie, "A26")

# Chart 4: Average review score by delivery status.
avg_chart = BarChart()
avg_chart.type = "col"
avg_chart.style = 11
avg_chart.title = "Average Review Score by Delivery Status"
avg_chart.y_axis.title = "Average score (out of 5)"
avg_chart.x_axis.title = "Delivery status"
avg_chart.height = 8.2
avg_chart.width = 17.5
avg_chart.legend = None

avg_data = Reference(
    delivery_ws,
    min_col=3,
    min_row=1,
    max_row=delivery_ws.max_row
)

avg_chart.add_data(avg_data, titles_from_data=True)
avg_chart.set_categories(delivery_categories)
avg_chart.dataLabels = DataLabelList()
avg_chart.dataLabels.showVal = True

dash.add_chart(avg_chart, "I26")

# Interpretation and caveat block beneath the charts.
dash.merge_cells("A43:P43")
dash["A43"] = "HOW TO READ THIS DASHBOARD"
dash["A43"].font = Font(name="Aptos", size=11, bold=True, color=white)
dash["A43"].fill = PatternFill("solid", fgColor=blue)
dash["A43"].alignment = Alignment(vertical="center")
dash.row_dimensions[43].height = 24

dash.merge_cells("A44:P46")
dash["A44"] = (
    "Delivery group counts include orders with both actual and estimated delivery dates. "
    "The review comparison is descriptive and does not establish that delivery timing alone "
    "caused a rating difference. Repeat purchasing means 2 or more distinct orders per "
    "customer_unique_id within the available dataset. September–October 2016 and "
    "September–October 2018 are partial months; November 2016 is absent."
)
dash["A44"].font = Font(name="Aptos", size=10, color=dark)
dash["A44"].alignment = Alignment(wrap_text=True, vertical="top")
dash["A44"].fill = PatternFill("solid", fgColor=pale)

# Set dashboard view and print layout.
dash.freeze_panes = "A4"
dash.sheet_view.zoomScale = 80
dash.sheet_properties.pageSetUpPr.fitToPage = True
dash.page_setup.orientation = "landscape"
dash.page_setup.fitToWidth = 1
dash.page_setup.fitToHeight = 2
dash.print_area = "A1:P46"

# Make Dashboard the first active sheet.
wb.active = 0

# Save as a new file so the uploaded original remains untouched.
dashboard_file = "PrateekKumar_DTU_Analysis_Dashboard.xlsx"
wb.save(dashboard_file)

print("Dashboard updated successfully.")
print("Saved workbook:", dashboard_file)

files.download(dashboard_file)

Upload your latest Excel workbook (.xlsx).


Saving PrateekKumar_DTU_Analysis_Dashboard.xlsx to PrateekKumar_DTU_Analysis_Dashboard (1).xlsx
Dashboard updated successfully.
Saved workbook: PrateekKumar_DTU_Analysis_Dashboard.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Create a one-page Action Plan PDF from your latest Excel analysis workbook.
# Run this in Google Colab.

!pip -q install openpyxl reportlab matplotlib

import os
import math
import textwrap
from io import BytesIO

import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from google.colab import files

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, KeepTogether
)

# ---------------------------------------------------------
# 1. Upload and read your latest workbook
# ---------------------------------------------------------

print("Upload your latest Excel workbook (.xlsx), including the Dashboard tab.")
uploaded = files.upload()

xlsx_name = next(
    (name for name in uploaded.keys() if name.lower().endswith(".xlsx")),
    None
)

if not xlsx_name:
    raise FileNotFoundError("Please upload an .xlsx workbook.")

wb = load_workbook(xlsx_name, data_only=True)

needed_sheets = ["Monthly Orders", "Delivery Reviews", "Customer Repeat"]
missing = [s for s in needed_sheets if s not in wb.sheetnames]
if missing:
    raise ValueError(
        f"Workbook is missing these required tabs: {missing}. "
        "Please upload the latest analysis workbook."
    )

# Read a sheet as a dataframe, using its first row as headers.
def sheet_to_df(sheet_name):
    ws = wb[sheet_name]
    values = list(ws.values)
    headers = [str(x).strip() if x is not None else "" for x in values[0]]
    df = pd.DataFrame(values[1:], columns=headers)
    df = df.dropna(how="all")
    return df

monthly_df = sheet_to_df("Monthly Orders")
delivery_df = sheet_to_df("Delivery Reviews")
repeat_df = sheet_to_df("Customer Repeat")

# Normalize column names for flexible matching.
def normalized_columns(df):
    return {str(c).strip().lower(): c for c in df.columns}

monthly_cols = normalized_columns(monthly_df)
delivery_cols = normalized_columns(delivery_df)
repeat_cols = normalized_columns(repeat_df)

def find_col(col_map, options, sheet_name):
    for option in options:
        if option.lower() in col_map:
            return col_map[option.lower()]
    raise ValueError(
        f"Could not find a required column in '{sheet_name}'. "
        f"Expected one of: {options}. Found: {list(col_map.keys())}"
    )

month_col = find_col(
    monthly_cols, ["month", "purchase_month"], "Monthly Orders"
)
monthly_count_col = find_col(
    monthly_cols, ["order_count", "orders"], "Monthly Orders"
)

status_col = find_col(
    delivery_cols, ["delivery_performance"], "Delivery Reviews"
)
review_count_col = find_col(
    delivery_cols,
    ["orders_with_review_score", "orders_with_reviews", "orders_with_delivery_dates"],
    "Delivery Reviews"
)
avg_score_col = find_col(
    delivery_cols, ["average_review_score"], "Delivery Reviews"
)
low_share_col = find_col(
    delivery_cols,
    ["low_rated_order_share_pct", "low_rated_share_pct"],
    "Delivery Reviews"
)

customer_type_col = find_col(
    repeat_cols, ["customer_type"], "Customer Repeat"
)
customer_count_col = find_col(
    repeat_cols, ["customers", "customer_count"], "Customer Repeat"
)
repeat_share_col = find_col(
    repeat_cols,
    ["share_of_customers_pct", "customer_share_pct"],
    "Customer Repeat"
)

# Clean and sort monthly series.
monthly_df[monthly_count_col] = pd.to_numeric(
    monthly_df[monthly_count_col], errors="coerce"
)
monthly_df = monthly_df.dropna(subset=[month_col, monthly_count_col]).copy()
monthly_df[month_col] = monthly_df[month_col].astype(str)
monthly_df = monthly_df.sort_values(month_col)

# Normalize delivery labels and numeric fields.
delivery_df[status_col] = delivery_df[status_col].astype(str).str.strip()
for col in [review_count_col, avg_score_col, low_share_col]:
    delivery_df[col] = pd.to_numeric(delivery_df[col], errors="coerce")

def delivery_row(label):
    matches = delivery_df[
        delivery_df[status_col].str.lower() == label.lower()
    ]
    if matches.empty:
        raise ValueError(f"Could not find '{label}' in Delivery Reviews.")
    return matches.iloc[0]

late = delivery_row("Late")
on_time = delivery_row("On time")

late_count = int(late[review_count_col])
on_time_count = int(on_time[review_count_col])
late_avg = float(late[avg_score_col])
on_time_avg = float(on_time[avg_score_col])
late_low = float(late[low_share_col])
on_time_low = float(on_time[low_share_col])

repeat_df[customer_count_col] = pd.to_numeric(
    repeat_df[customer_count_col], errors="coerce"
)
repeat_df[repeat_share_col] = pd.to_numeric(
    repeat_df[repeat_share_col], errors="coerce"
)

def repeat_row(label):
    matches = repeat_df[
        repeat_df[customer_type_col].astype(str).str.lower() == label.lower()
    ]
    if matches.empty:
        raise ValueError(f"Could not find '{label}' in Customer Repeat.")
    return matches.iloc[0]

one_time = repeat_row("One-time")
repeat = repeat_row("Repeat")

one_time_count = int(one_time[customer_count_col])
repeat_count = int(repeat[customer_count_col])
repeat_pct = float(repeat[repeat_share_col])

# ---------------------------------------------------------
# 2. Create charts with permanent labels
# ---------------------------------------------------------

chart_buffers = []

def chart_buffer(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=180, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    chart_buffers.append(buf)
    return buf

# Chart A: Monthly order trend.
fig, ax = plt.subplots(figsize=(4.0, 1.65))
x = list(range(len(monthly_df)))
y = monthly_df[monthly_count_col].tolist()
ax.plot(x, y, marker="o", linewidth=2, markersize=2.5)
ax.set_title("Monthly Order Volume", fontsize=9, pad=5)
ax.set_ylabel("Orders", fontsize=7)
ax.set_xlabel("Purchase month", fontsize=7)
ax.grid(axis="y", alpha=0.25)
tick_positions = list(range(0, len(monthly_df), 3))
if len(monthly_df) - 1 not in tick_positions:
    tick_positions.append(len(monthly_df) - 1)
ax.set_xticks(tick_positions)
ax.set_xticklabels(
    [monthly_df.iloc[i][month_col] for i in tick_positions],
    rotation=35, ha="right", fontsize=6
)
ax.tick_params(axis="y", labelsize=6)
fig.tight_layout()
monthly_chart = chart_buffer(fig)

# Chart B: Average review score by delivery performance.
fig, ax = plt.subplots(figsize=(3.2, 1.65))
labels = ["Late", "On time"]
scores = [late_avg, on_time_avg]
bars = ax.bar(labels, scores)
ax.set_title("Average Review Score", fontsize=9, pad=5)
ax.set_ylabel("Score out of 5", fontsize=7)
ax.set_ylim(0, 5.6)
ax.tick_params(axis="both", labelsize=7)
ax.grid(axis="y", alpha=0.25)
for bar, val in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.10,
        f"{val:.2f}",
        ha="center", va="bottom", fontsize=7
    )
fig.tight_layout()
review_chart = chart_buffer(fig)

# Chart C: One-time versus repeat customer mix.
fig, ax = plt.subplots(figsize=(3.2, 1.65))
mix_labels = ["One-time", "Repeat"]
mix_counts = [one_time_count, repeat_count]
bars = ax.bar(mix_labels, mix_counts)
ax.set_title("Observed Customer Mix", fontsize=9, pad=5)
ax.set_ylabel("Customers", fontsize=7)
ax.tick_params(axis="both", labelsize=7)
ax.grid(axis="y", alpha=0.25)
total_cust = one_time_count + repeat_count
for bar, count in zip(bars, mix_counts):
    pct = count / total_cust * 100 if total_cust else 0
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(mix_counts) * 0.02,
        f"{count:,}\n({pct:.2f}%)",
        ha="center", va="bottom", fontsize=6.5
    )
fig.tight_layout()
repeat_chart = chart_buffer(fig)

# ---------------------------------------------------------
# 3. Build the one-page PDF
# ---------------------------------------------------------

output_pdf = "PrateekKumar_DTU_ActionPlan.pdf"

page_size = landscape(A4)
page_width, page_height = page_size

doc = SimpleDocTemplate(
    output_pdf,
    pagesize=page_size,
    rightMargin=10 * mm,
    leftMargin=10 * mm,
    topMargin=7 * mm,
    bottomMargin=7 * mm
)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "TitleCustom", parent=styles["Title"],
    fontName="Helvetica-Bold", fontSize=16, leading=18,
    alignment=TA_CENTER, textColor=colors.HexColor("#17365D"),
    spaceAfter=2
)
subtitle_style = ParagraphStyle(
    "SubtitleCustom", parent=styles["Normal"],
    fontName="Helvetica", fontSize=8.5, leading=10,
    alignment=TA_CENTER, textColor=colors.HexColor("#4B5563"),
    spaceAfter=5
)
section_style = ParagraphStyle(
    "SectionCustom", parent=styles["Heading2"],
    fontName="Helvetica-Bold", fontSize=10, leading=12,
    textColor=colors.HexColor("#17365D"),
    spaceBefore=3, spaceAfter=3
)
body_style = ParagraphStyle(
    "BodyCustom", parent=styles["BodyText"],
    fontName="Helvetica", fontSize=7.5, leading=9,
    alignment=TA_LEFT
)
small_style = ParagraphStyle(
    "SmallCustom", parent=body_style,
    fontSize=6.7, leading=8
)
table_header_style = ParagraphStyle(
    "TableHeaderCustom", parent=body_style,
    fontName="Helvetica-Bold", fontSize=7.2, leading=8.2,
    textColor=colors.white
)
table_cell_style = ParagraphStyle(
    "TableCellCustom", parent=body_style,
    fontSize=7.0, leading=8.2
)
priority_style = ParagraphStyle(
    "PriorityCustom", parent=table_cell_style,
    fontName="Helvetica-Bold", textColor=colors.HexColor("#17365D")
)

def P(text, style=body_style):
    return Paragraph(text, style)

story = []

story.append(P("Brazilian E-Commerce Marketplace | Action Plan", title_style))
story.append(P(
    "Objective: improve customer experience, investigate uneven review scores "
    "and support sustainable repeat purchasing.",
    subtitle_style
))

story.append(P("Key evidence from the analysis", section_style))

evidence_data = [
    [
        P("<b>Order volume</b>", table_cell_style),
        P(
            "Monthly orders rose from 800 in January 2017 to 7,544 in November 2017. "
            "The latest observed months include partial periods, so the trend needs "
            "careful interpretation.",
            table_cell_style
        )
    ],
    [
        P("<b>Delivery and ratings</b>", table_cell_style),
        P(
            f"Orders delivered late had an average review score of {late_avg:.2f}, "
            f"compared with {on_time_avg:.2f} for on-time orders. Low-rating shares "
            f"were {late_low:.1f}% and {on_time_low:.1f}%, respectively.",
            table_cell_style
        )
    ],
    [
        P("<b>Repeat purchasing</b>", table_cell_style),
        P(
            f"{repeat_count:,} of {total_cust:,} unique customers were observed "
            f"placing more than one order ({repeat_pct:.2f}%). This is a historical "
            "dataset measure, not a forecast of future retention.",
            table_cell_style
        )
    ]
]

evidence_table = Table(
    evidence_data,
    colWidths=[36 * mm, 232 * mm],
    hAlign="LEFT"
)
evidence_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#DCEAF5")),
    ("BACKGROUND", (1, 0), (1, -1), colors.HexColor("#F7F9FC")),
    ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C7D1DC")),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("LEFTPADDING", (0, 0), (-1, -1), 5),
    ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ("TOPPADDING", (0, 0), (-1, -1), 3),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
]))
story.append(evidence_table)
story.append(Spacer(1, 3))

# Three charts in a single row, with labels rendered on the charts.
story.append(P("Analysis snapshot", section_style))

chart_row = Table(
    [[
        Image(monthly_chart, width=90 * mm, height=37 * mm),
        Image(review_chart, width=72 * mm, height=37 * mm),
        Image(repeat_chart, width=72 * mm, height=37 * mm)
    ]],
    colWidths=[94 * mm, 76 * mm, 76 * mm],
    hAlign="LEFT"
)
chart_row.setStyle(TableStyle([
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("LEFTPADDING", (0, 0), (-1, -1), 1),
    ("RIGHTPADDING", (0, 0), (-1, -1), 1),
    ("TOPPADDING", (0, 0), (-1, -1), 0),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
]))
story.append(chart_row)
story.append(Spacer(1, 2))

story.append(P("Prioritized recommendations", section_style))

recommendations = [
    [
        P("Priority", table_header_style),
        P("Recommendation and next action", table_header_style),
        P("Impact and effort rationale", table_header_style),
        P("KPIs to monitor", table_header_style)
    ],
    [
        P("1", priority_style),
        P(
            "<b>Diagnose variation by seller, region and product category.</b> "
            "Compare lateness and review scores across these groups, check sample "
            "sizes and data gaps, then validate the main patterns with operations.",
            table_cell_style
        ),
        P(
            "<b>High impact, medium effort.</b> Directly addresses leadership's "
            "uncertainty about the drivers of uneven customer experience; requires "
            "joined data and validation.",
            table_cell_style
        ),
        P(
            "On-time delivery rate; average review score; low-rating share; "
            "late-delivery rate by seller, region and category.",
            table_cell_style
        )
    ],
    [
        P("2", priority_style),
        P(
            "<b>Pilot targeted delivery improvements.</b> After the diagnostic, "
            "investigate dispatch delays, carrier handoffs and delivery estimates "
            "in selected problem areas. Test changes on a limited scale before rollout.",
            table_cell_style
        ),
        P(
            "<b>High impact, medium effort.</b> Delivery status is associated with "
            "a sizeable rating difference in the analysis; a pilot tests whether "
            "operational changes improve outcomes.",
            table_cell_style
        ),
        P(
            "Pilot on-time delivery rate; late-order share; average review score; "
            "low-rating share; actual versus estimated delivery time.",
            table_cell_style
        )
    ],
    [
        P("3", priority_style),
        P(
            "<b>Test a customer retention initiative.</b> Track repeat ordering by "
            "first-purchase cohort. Compare a follow-up message or second-order "
            "offer with a similar group that does not receive it.",
            table_cell_style
        ),
        P(
            "<b>Medium to high impact, medium effort.</b> The observed repeat share "
            "does not explain why customers do or do not return; a controlled test "
            "can measure response.",
            table_cell_style
        ),
        P(
            "Repeat purchase rate by cohort; second-order conversion; time to "
            "second order; incremental repeat orders; contribution margin after offer cost.",
            table_cell_style
        )
    ]
]

action_table = Table(
    recommendations,
    colWidths=[15 * mm, 105 * mm, 72 * mm, 54 * mm],
    repeatRows=1,
    hAlign="LEFT"
)
action_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#17365D")),
    ("BACKGROUND", (0, 1), (0, -1), colors.HexColor("#DCEAF5")),
    ("BACKGROUND", (1, 1), (-1, -1), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C7D1DC")),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("LEFTPADDING", (0, 0), (-1, -1), 4),
    ("RIGHTPADDING", (0, 0), (-1, -1), 4),
    ("TOPPADDING", (0, 0), (-1, -1), 4),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
]))
story.append(action_table)
story.append(Spacer(1, 4))

story.append(P("Execution sequence", section_style))

execution_data = [
    [
        P("<b>First 30 days</b>", table_cell_style),
        P(
            "Complete the seller, region and category diagnostic. Establish KPI "
            "baselines and select priority areas for a delivery pilot.",
            table_cell_style
        )
    ],
    [
        P("<b>Days 31 to 60</b>", table_cell_style),
        P(
            "Run the focused delivery pilot and customer retention test. Compare "
            "results with baseline and control groups before deciding on expansion.",
            table_cell_style
        )
    ]
]
execution_table = Table(
    execution_data,
    colWidths=[36 * mm, 232 * mm],
    hAlign="LEFT"
)
execution_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#DCEAF5")),
    ("BACKGROUND", (1, 0), (1, -1), colors.HexColor("#F7F9FC")),
    ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C7D1DC")),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("LEFTPADDING", (0, 0), (-1, -1), 5),
    ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ("TOPPADDING", (0, 0), (-1, -1), 3),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
]))
story.append(execution_table)
story.append(Spacer(1, 3))

story.append(P(
    "<b>Interpretation note:</b> These are descriptive historical patterns. "
    "They do not establish that late delivery causes lower ratings or that any "
    "proposed action will increase repeat purchasing. Validate the drivers and "
    "measure pilot outcomes before scaling.",
    small_style
))

# Create PDF.
doc.build(story)

print(f"PDF created: {output_pdf}")
files.download(output_pdf)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.7 MB/s eta 0:00:00
Upload your latest Excel workbook (.xlsx), including the Dashboard tab.


Saving PrateekKumar_DTU_Analysis.xlsx to PrateekKumar_DTU_Analysis (1).xlsx
PDF created: PrateekKumar_DTU_ActionPlan.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Olist case assessment: create an improved one-page Action Plan PDF
# Run this cell in Google Colab and upload your latest .xlsx workbook.

!pip -q install openpyxl reportlab matplotlib pypdf

import os
from io import BytesIO

import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from google.colab import files

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
)
from pypdf import PdfReader

# ------------------------------------------------------------
# 1. Upload the latest workbook and read the analysis sheets
# ------------------------------------------------------------

print("Upload your latest Excel analysis workbook (.xlsx).")
uploaded = files.upload()

xlsx_name = next(
    (name for name in uploaded if name.lower().endswith(".xlsx")),
    None
)
if not xlsx_name:
    raise ValueError("Please upload your latest Excel workbook in .xlsx format.")

wb = load_workbook(xlsx_name, data_only=True)

required_sheets = ["Monthly Orders", "Delivery Reviews", "Customer Repeat"]
missing_sheets = [s for s in required_sheets if s not in wb.sheetnames]
if missing_sheets:
    raise ValueError(f"Workbook is missing required sheets: {missing_sheets}")

def read_sheet(sheet_name):
    ws = wb[sheet_name]
    rows = list(ws.values)
    if not rows:
        raise ValueError(f"Sheet '{sheet_name}' is empty.")
    headers = [
        str(value).strip() if value is not None else ""
        for value in rows[0]
    ]
    df = pd.DataFrame(rows[1:], columns=headers).dropna(how="all")
    return df

monthly = read_sheet("Monthly Orders")
delivery = read_sheet("Delivery Reviews")
customers = read_sheet("Customer Repeat")

def find_column(df, candidates, sheet_name):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise ValueError(
        f"Could not find a column in '{sheet_name}'. "
        f"Expected one of {candidates}. Found: {list(df.columns)}"
    )

# Monthly Orders columns
month_col = find_column(
    monthly, ["month", "purchase_month"], "Monthly Orders"
)
monthly_orders_col = find_column(
    monthly, ["order_count", "orders"], "Monthly Orders"
)

# Delivery Reviews columns
status_col = find_column(
    delivery, ["delivery_performance"], "Delivery Reviews"
)
delivery_count_col = find_column(
    delivery,
    [
        "orders_with_review_score",
        "orders_with_reviews",
        "orders_with_delivery_dates"
    ],
    "Delivery Reviews"
)
avg_score_col = find_column(
    delivery, ["average_review_score"], "Delivery Reviews"
)
low_share_col = find_column(
    delivery,
    ["low_rated_order_share_pct", "low_rated_share_pct"],
    "Delivery Reviews"
)

# Customer Repeat columns
customer_type_col = find_column(
    customers, ["customer_type"], "Customer Repeat"
)
customer_count_col = find_column(
    customers, ["customers", "customer_count"], "Customer Repeat"
)
customer_share_col = find_column(
    customers,
    ["share_of_customers_pct", "customer_share_pct"],
    "Customer Repeat"
)

# Convert values to numeric and remove blank rows.
monthly[monthly_orders_col] = pd.to_numeric(
    monthly[monthly_orders_col], errors="coerce"
)
monthly = monthly.dropna(
    subset=[month_col, monthly_orders_col]
).copy()
monthly[month_col] = monthly[month_col].astype(str)
monthly = monthly.sort_values(month_col)

for col in [delivery_count_col, avg_score_col, low_share_col]:
    delivery[col] = pd.to_numeric(delivery[col], errors="coerce")

customers[customer_count_col] = pd.to_numeric(
    customers[customer_count_col], errors="coerce"
)
customers[customer_share_col] = pd.to_numeric(
    customers[customer_share_col], errors="coerce"
)

def get_group(df, key_col, key_value, sheet_name):
    found = df[
        df[key_col].astype(str).str.strip().str.lower()
        == key_value.lower()
    ]
    if found.empty:
        raise ValueError(
            f"Could not find '{key_value}' in sheet '{sheet_name}'."
        )
    return found.iloc[0]

late = get_group(delivery, status_col, "Late", "Delivery Reviews")
on_time = get_group(delivery, status_col, "On time", "Delivery Reviews")
repeat = get_group(customers, customer_type_col, "Repeat", "Customer Repeat")
one_time = get_group(customers, customer_type_col, "One-time", "Customer Repeat")

late_n = int(late[delivery_count_col])
on_time_n = int(on_time[delivery_count_col])
late_avg = float(late[avg_score_col])
on_time_avg = float(on_time[avg_score_col])

def pct_value(value):
    """Handle a percentage stored as either 54.6 or 0.546."""
    value = float(value)
    return value * 100 if abs(value) <= 1 else value

late_low_pct = pct_value(late[low_share_col])
on_time_low_pct = pct_value(on_time[low_share_col])

repeat_n = int(repeat[customer_count_col])
one_time_n = int(one_time[customer_count_col])
repeat_pct = pct_value(repeat[customer_share_col])
total_delivery_orders = late_n + on_time_n
total_customers = repeat_n + one_time_n

# Basic data checks to catch unexpected workbook edits.
if total_delivery_orders <= 0 or total_customers <= 0:
    raise ValueError("One or more summary totals are zero or missing.")

# ------------------------------------------------------------
# 2. Make charts with clear, non-overlapping labels
# ------------------------------------------------------------

def save_chart(fig):
    buf = BytesIO()
    fig.savefig(
        buf, format="png", dpi=180, bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)
    buf.seek(0)
    return buf

# Chart 1: Monthly orders.
# Only selected months are labelled on the axis; no labels are piled
# on top of every line point.
fig, ax = plt.subplots(figsize=(5.0, 1.65))

x = list(range(len(monthly)))
y = monthly[monthly_orders_col].astype(float).tolist()

ax.plot(x, y, marker="o", linewidth=1.8, markersize=2.8)
ax.set_title("Monthly Order Volume", fontsize=9, pad=5)
ax.set_ylabel("Orders", fontsize=7)
ax.set_xlabel("Purchase month", fontsize=7)
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="y", labelsize=6)

tick_step = max(1, (len(monthly) + 7) // 8)
ticks = list(range(0, len(monthly), tick_step))
if len(monthly) - 1 not in ticks:
    ticks.append(len(monthly) - 1)

ax.set_xticks(ticks)
ax.set_xticklabels(
    [monthly.iloc[i][month_col] for i in ticks],
    rotation=30, ha="right", fontsize=6
)
fig.tight_layout()
monthly_chart = save_chart(fig)

# Chart 2: Review scores.
fig, ax = plt.subplots(figsize=(3.0, 1.65))
labels = ["Late", "On time"]
scores = [late_avg, on_time_avg]
bars = ax.bar(labels, scores)

ax.set_title("Average Review Score", fontsize=9, pad=5)
ax.set_ylabel("Score out of 5", fontsize=7)
ax.set_ylim(0, 5.5)
ax.tick_params(axis="both", labelsize=7)
ax.grid(axis="y", alpha=0.25)

for bar, score in zip(bars, scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.10,
        f"{score:.2f}",
        ha="center", va="bottom", fontsize=7
    )

fig.tight_layout()
review_chart = save_chart(fig)

# Chart 3: Customer mix.
# Horizontal bars provide room for the exact counts and percentages.
fig, ax = plt.subplots(figsize=(3.3, 1.65))
mix_labels = ["One-time", "Repeat"]
mix_counts = [one_time_n, repeat_n]
mix_pcts = [
    one_time_n / total_customers * 100,
    repeat_n / total_customers * 100
]
bars = ax.barh(mix_labels, mix_counts)

ax.set_title("Observed Customer Mix", fontsize=9, pad=5)
ax.set_xlabel("Unique customers", fontsize=7)
ax.tick_params(axis="both", labelsize=7)
ax.grid(axis="x", alpha=0.25)
ax.invert_yaxis()
ax.set_xlim(0, max(mix_counts) * 1.28)

for bar, count, pct in zip(bars, mix_counts, mix_pcts):
    ax.text(
        bar.get_width() + max(mix_counts) * 0.025,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,}  ({pct:.2f}%)",
        va="center", ha="left", fontsize=6.5
    )

fig.tight_layout()
customer_chart = save_chart(fig)

# ------------------------------------------------------------
# 3. PDF styling
# ------------------------------------------------------------

output_pdf = "PrateekKumar_DTU_ActionPlan.pdf"

page_size = landscape(A4)
doc = SimpleDocTemplate(
    output_pdf,
    pagesize=page_size,
    leftMargin=9 * mm,
    rightMargin=9 * mm,
    topMargin=6 * mm,
    bottomMargin=6 * mm
)

navy = colors.HexColor("#17365D")
pale_blue = colors.HexColor("#DCEAF5")
pale_grey = colors.HexColor("#F7F9FC")
grid_grey = colors.HexColor("#C7D1DC")
dark_grey = colors.HexColor("#4B5563")

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "PlanTitle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=15,
    leading=17,
    alignment=TA_CENTER,
    textColor=navy,
    spaceAfter=2
)

subtitle_style = ParagraphStyle(
    "PlanSubtitle",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=8,
    leading=9.5,
    alignment=TA_CENTER,
    textColor=dark_grey,
    spaceAfter=4
)

section_style = ParagraphStyle(
    "SectionHeading",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=9.2,
    leading=10.5,
    textColor=navy,
    spaceBefore=2,
    spaceAfter=2
)

body_style = ParagraphStyle(
    "BodyTextSmall",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.0,
    leading=8.2,
    alignment=TA_LEFT
)

cell_style = ParagraphStyle(
    "TableCell",
    parent=body_style,
    fontSize=6.7,
    leading=7.8
)

header_style = ParagraphStyle(
    "TableHeader",
    parent=body_style,
    fontName="Helvetica-Bold",
    fontSize=6.8,
    leading=7.8,
    textColor=colors.white
)

priority_style = ParagraphStyle(
    "PriorityCell",
    parent=cell_style,
    fontName="Helvetica-Bold",
    textColor=navy
)

small_style = ParagraphStyle(
    "Footnote",
    parent=body_style,
    fontSize=6.2,
    leading=7.2,
    textColor=dark_grey
)

def para(text, style=body_style):
    return Paragraph(text, style)

story = []

# Header and decision request.
story.append(para(
    "Olist E-commerce Marketplace | Action Plan",
    title_style
))
story.append(para(
    "Objective: improve customer experience, investigate uneven review scores "
    "and support sustainable repeat purchasing.",
    subtitle_style
))

decision_box = Table(
    [[para(
        "<b>Decision requested</b> &nbsp; Approve a short diagnostic phase to "
        "identify where delivery and review performance vary, then use the "
        "results to select a focused operational pilot.",
        body_style
    )]],
    colWidths=[279 * mm]
)
decision_box.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, -1), pale_blue),
    ("BOX", (0, 0), (-1, -1), 0.5, grid_grey),
    ("LEFTPADDING", (0, 0), (-1, -1), 6),
    ("RIGHTPADDING", (0, 0), (-1, -1), 6),
    ("TOPPADDING", (0, 0), (-1, -1), 4),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
]))
story.append(decision_box)
story.append(Spacer(1, 3))

# Evidence summary.
story.append(para("Key evidence from the analysis", section_style))

evidence_rows = [
    [
        para("<b>Order volume</b>", cell_style),
        para(
            "Monthly order counts increased during the observed period. "
            "The earliest and latest months may be partial periods, so avoid "
            "direct comparisons with complete months.",
            cell_style
        )
    ],
    [
        para("<b>Delivery and ratings</b>", cell_style),
        para(
            f"Among {total_delivery_orders:,} orders with a scored review and "
            f"both delivery dates, late orders averaged {late_avg:.2f}/5, "
            f"compared with {on_time_avg:.2f}/5 for on-time orders. "
            f"Low-rating shares were {late_low_pct:.1f}% and "
            f"{on_time_low_pct:.1f}%, respectively.",
            cell_style
        )
    ],
    [
        para("<b>Repeat purchasing</b>", cell_style),
        para(
            f"{repeat_n:,} of {total_customers:,} unique customers were observed "
            f"placing more than one order ({repeat_pct:.2f}%). This is a "
            "historical observed share, not a forecast of future retention.",
            cell_style
        )
    ]
]

evidence_table = Table(
    evidence_rows,
    colWidths=[34 * mm, 245 * mm],
    hAlign="LEFT"
)
evidence_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (0, -1), pale_blue),
    ("BACKGROUND", (1, 0), (1, -1), pale_grey),
    ("GRID", (0, 0), (-1, -1), 0.35, grid_grey),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("LEFTPADDING", (0, 0), (-1, -1), 5),
    ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ("TOPPADDING", (0, 0), (-1, -1), 3),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
]))
story.append(evidence_table)
story.append(Spacer(1, 2))

# Charts with labels visible without hovering.
story.append(para("Analysis snapshot", section_style))

chart_table = Table(
    [[
        Image(monthly_chart, width=110 * mm, height=37 * mm),
        Image(review_chart, width=73 * mm, height=37 * mm),
        Image(customer_chart, width=78 * mm, height=37 * mm)
    ]],
    colWidths=[115 * mm, 77 * mm, 87 * mm],
    hAlign="LEFT"
)
chart_table.setStyle(TableStyle([
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("LEFTPADDING", (0, 0), (-1, -1), 1),
    ("RIGHTPADDING", (0, 0), (-1, -1), 1),
    ("TOPPADDING", (0, 0), (-1, -1), 0),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
]))
story.append(chart_table)
story.append(Spacer(1, 2))

# Recommendations.
story.append(para("Prioritized recommendations", section_style))

recommendation_rows = [
    [
        para("Priority", header_style),
        para("Recommendation and next action", header_style),
        para("Impact and effort rationale", header_style),
        para("KPIs to monitor", header_style)
    ],
    [
        para("1", priority_style),
        para(
            "<b>Diagnose variation by seller, region and product category.</b> "
            "Compare delivery lateness and review scores across these groups. "
            "Check sample sizes and missing data, then validate key patterns "
            "with the operations team.",
            cell_style
        ),
        para(
            "<b>High impact, medium effort.</b> Addresses the open question of "
            "what may drive uneven customer experience. Requires joined data "
            "and validation.",
            cell_style
        ),
        para(
            "On-time delivery rate; average review score; low-rating share; "
            "late-delivery rate by seller, region and category.",
            cell_style
        )
    ],
    [
        para("2", priority_style),
        para(
            "<b>Pilot targeted delivery improvements.</b> After the diagnostic, "
            "investigate dispatch delays, carrier handoffs and delivery estimates "
            "in selected problem areas. Test changes on a limited scale before "
            "considering wider rollout.",
            cell_style
        ),
        para(
            "<b>High impact, medium effort.</b> Delivery status is associated "
            "with a substantial rating difference in this analysis. The pilot "
            "will test whether operational changes improve outcomes.",
            cell_style
        ),
        para(
            "Pilot on-time delivery rate; late-order share; average review score; "
            "low-rating share; actual versus estimated delivery time.",
            cell_style
        )
    ],
    [
        para("3", priority_style),
        para(
            "<b>Test a customer retention initiative.</b> Track repeat orders by "
            "first-purchase cohort. Compare a follow-up message or second-order "
            "offer with a similar group that does not receive it.",
            cell_style
        ),
        para(
            "<b>Medium to high impact, medium effort.</b> The observed repeat "
            "share does not explain why customers do or do not return. A "
            "controlled test can measure response.",
            cell_style
        ),
        para(
            "Repeat purchase rate by cohort; second-order conversion; time to "
            "second order; incremental repeat orders; contribution margin after "
            "offer cost.",
            cell_style
        )
    ]
]

recommendation_table = Table(
    recommendation_rows,
    colWidths=[15 * mm, 104 * mm, 77 * mm, 83 * mm],
    repeatRows=1,
    hAlign="LEFT"
)
recommendation_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), navy),
    ("BACKGROUND", (0, 1), (0, -1), pale_blue),
    ("BACKGROUND", (1, 1), (-1, -1), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.35, grid_grey),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("LEFTPADDING", (0, 0), (-1, -1), 4),
    ("RIGHTPADDING", (0, 0), (-1, -1), 4),
    ("TOPPADDING", (0, 0), (-1, -1), 3),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
]))
story.append(recommendation_table)
story.append(Spacer(1, 3))

# Sequence and methodology note.
story.append(para("Execution sequence", section_style))

execution_rows = [
    [
        para("<b>First 30 days</b>", cell_style),
        para(
            "Complete the seller, region and category diagnostic. Establish "
            "baseline KPIs and select a focused area for a delivery pilot.",
            cell_style
        ),
        para("<b>Days 31 to 60</b>", cell_style),
        para(
            "Run the delivery pilot and retention test. Compare results with "
            "baseline and comparison groups before deciding whether to expand.",
            cell_style
        )
    ]
]

execution_table = Table(
    execution_rows,
    colWidths=[28 * mm, 112 * mm, 28 * mm, 111 * mm],
    hAlign="LEFT"
)
execution_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (0, 0), pale_blue),
    ("BACKGROUND", (2, 0), (2, 0), pale_blue),
    ("BACKGROUND", (1, 0), (1, 0), pale_grey),
    ("BACKGROUND", (3, 0), (3, 0), pale_grey),
    ("GRID", (0, 0), (-1, -1), 0.35, grid_grey),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("LEFTPADDING", (0, 0), (-1, -1), 4),
    ("RIGHTPADDING", (0, 0), (-1, -1), 4),
    ("TOPPADDING", (0, 0), (-1, -1), 3),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
]))
story.append(execution_table)
story.append(Spacer(1, 2))

story.append(para(
    "<b>Measurement note:</b> The delivery comparison covers orders with a "
    "scored review and both actual and estimated delivery dates. Low rating "
    "means an order-level review score of 1 or 2, based on the workbook's "
    "calculation. Findings are descriptive and do not establish causation. "
    "Source: supplied Olist datasets, 2016 to 2018. Historical data may include "
    "partial months and missing fields.",
    small_style
))

# ------------------------------------------------------------
# 4. Build, verify page count and download
# ------------------------------------------------------------

doc.build(story)

page_count = len(PdfReader(output_pdf).pages)
print(f"PDF created: {output_pdf}")
print(f"Number of pages: {page_count}")

if page_count != 1:
    print(
        "Note: The PDF is not one page. Reduce font sizes or chart heights "
        "slightly, then rerun the cell."
    )

files.download(output_pdf)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 17.3 MB/s eta 0:00:00
Upload your latest Excel analysis workbook (.xlsx).


Saving PrateekKumar_DTU_Analysis.xlsx to PrateekKumar_DTU_Analysis (2).xlsx
PDF created: PrateekKumar_DTU_ActionPlan.pdf
Number of pages: 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>